In [1]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.decks.LoadAll import *
from science_jubilee.tools.HTTPSyringe import *
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *
#from science_jubilee.tools.Detection_erreur import Detection_erreur
from science_jubilee.tools.Loop import *

In [2]:
# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=True)
ino = Loop
#detect = Detection_erreur
ser = HTTPSyringe

2026-04-08 10:47:43 - [JubileeController]  - INFO - Initializing JubileeController (simulated=True, address=10.0.9.6)
2026-04-08 10:47:43 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-04-08 10:47:43 - [JubileeController]  - INFO - Running in simulation mode. No connection established.
2026-04-08 10:47:43 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=True, max_tools=4).


In [3]:
# Homing
jm.controller.home_all()

Is a tool currently mounted? [y/n]  n
Is the deck clear of any obstacles? [y/n]  y


In [4]:
execute_plan('plan_jubilee', jm=jm)
#trace le plan

🚀 Exécution du plan Jubilee : plan_jubilee.txt
✅ Plan terminé avec succès sur Jubilee.


In [5]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=300)
jm.controller.move_to(x=150, y=150)

2026-04-08 10:47:51 - [JubileeController]  - INFO - (SIMULATED) move_to(x=None, y=None, z=300, u=None, s=6000, param=None, wait=False)
2026-04-08 10:47:51 - [JubileeController]  - INFO - (SIMULATED) move_to(x=150, y=150, z=None, u=None, s=6000, param=None, wait=False)


In [6]:
# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

2026-04-08 10:47:52 - [Deck]               - INFO - Loading deck configuration from: D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition\test1.json
2026-04-08 10:47:52 - [Deck]               - INFO - Deck 'Experience1' loaded with 3 slots.
2026-04-08 10:47:52 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from 'D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition'.
2026-04-08 10:47:52 - [Tool]               - INFO - Tool 'Inoculator' (index 0) initialized.
2026-04-08 10:47:52 - [JubileeManager]     - INFO - Tool 'Inoculator' loaded at index 0.
2026-04-08 10:47:52 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, 36, 0).
2026-04-08 10:47:52 - [Tool]               - INFO - Tool 'Pipette' (index 1) initialized.
2026-04-08 10:47:52 - [JubileeManager]     - INFO - Tool 'Pipette' loaded at index 1.
2026-04-08 10:47:52 - [JubileeManager]     - INFO - Offset for tool at index 0 set t

In [7]:
#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

{'0': (48.45, 95.26), '1': (165.75, 52.75), '2': (167.08, 193.75)}

In [8]:
jm.status()

{'deck': 'Experience1',
 'tools': [{'index': 0, 'name': 'Inoculator'},
  {'index': 1, 'name': 'Pipette'},
  {'index': 2, 'name': 'stylo'}],
 'active_tool_index': None,
 'active_tool_name': None,
 'tool_offsets': {0: (0, 36, 0)}}

In [9]:
#index de la pipette
dict_outil = jm.get_tool_by_name("Pipette")
idx_pipette = dict_outil['index']

jm.controller.pickup_tool_sequence(idx_pipette)
jm.set_active_tool(idx_pipette)
jm.set_tool_offset(idx_pipette, (0, -31.5, 35))


2026-04-08 10:48:40 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 1.
2026-04-08 10:48:40 - [JubileeController]  - INFO - (SIMULATED) pickup_tool_sequence(index=1, z_park=None)
2026-04-08 10:48:40 - [JubileeManager]     - INFO - Tool at index 1 set as active tool.
2026-04-08 10:48:40 - [JubileeManager]     - INFO - Offset for tool at index 1 set to (0, -31.5, 35).


In [ ]:
#  1. Se connecter à la Raspberry Pi (ssh jubilee@10.0.9.55) mdp : projet_indus

#  2. Lancer le serveur avec la commande : python3 serveur_pi.py

#  3. Vérifier que l'IP est bien 10.0.9.55 et le port 5001

In [10]:
pipette = jm.tools_list[idx_pipette]
pipette.__class__ = ser
pipette.url_materiel = "http://10.0.9.55:5001" 
pipette._init_gpio()
pipette._machine = jm.controller  # On le lie au contrôleur pour les mouvements
pipette.is_active_tool = True     # On valide le décorateur de sécurité

#On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 220mm de hauteur pour ne rien toucher
    pipette.safe_z_movement = lambda: jm.controller.move_to(z=220)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{pipette.name}' prêt (Classe: {type(pipette).__name__})")

[Pipette] ✅ Connecté au serveur matériel du Raspberry Pi.
✅ Correctif safe_z_movement appliqué.
✅ Outil 'Pipette' prêt (Classe: HTTPSyringe)


In [14]:
eau = None  # Initialisation pour éviter une erreur au print final
inoculateur=None
puits=None

all_slots = jm.deck.list_slots() 

for slot_id in all_slots:
    slot = jm.deck.get_slot(slot_id)
    labware_name = str(slot.labware) # On force en string au cas où c'est un objet
    if "pot_de_d'eau" in labware_name:
        eau = slot
        print(f"---> EAU TROUVÉ au slot {slot_id} !")
        
    if "pot_duckweed" in labware_name:
        duckweed = slot
        print(f"---> DUCKWEED TROUVÉ au slot {slot_id} !")
        
    if "greiner" in labware_name:
        puits = slot
        print(f"---> PUITS TROUVÉ au slot {slot_id} !")

---> PUITS TROUVÉ au slot 0 !
---> EAU TROUVÉ au slot 1 !
---> DUCKWEED TROUVÉ au slot 2 !


In [12]:
well_source = eau.labware.wells['A1']
well_source.y = well_source.y + eau.coordinates[1] - 50 # ac offset de x
well_source.x = well_source.x + eau.coordinates[0] - 5.5 #ac offset de y
#print(well_source.x)
#print(well_source.y)

In [37]:
# offset pipette
# x = -5.5
# y = -50
# z = 121 pour vider

cpt = 0 
# --- CONFIGURATION DES PUITS ---
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)


jm.controller.move_to(x=150, y=150)
jm.controller.move_to(z=220)

print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        if ( cpt == 0 ):
            jm.controller.move_to(z=220)
            jm.controller.move_to(x=well_source.x,y=well_source.y)
            jm.controller.move_to(z=110)
            try:
                jm.controller.dwell(3000)
                pipette.remplir_seringue(temps_secondes=10.0)
            except Exception as e:
                print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
                break
            jm.controller.move_to(z=220)
            cpt = 3
            
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6
        
        # On récupère l'objet Well de destination
        well_dest = puits.labware.wells[well_dest_name]
        well_dest.y = (well_dest.y + puits.coordinates[1]) - 50
        well_dest.x = (well_dest.x + puits.coordinates[0]) - 5.5

        print(f"\n---> Transfert de {well_source.name} vers {well_dest_name}...")

        jm.controller.move_to(z=220)
        jm.controller.move_to(x=well_dest.x,y=well_dest.y)
        jm.controller.move_to(z=121)
        try:
            jm.controller.dwell(3000)
            pipette.avancer_jusqu_au_seuil(seuil=1, timeout_sec=5)
        except Exception as e:
            print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
            break
        cpt = cpt - 1
        jm.controller.move_to(z=220)

print("\n✅ Distribution terminée sur toute la plaque !")

2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) move_to(x=150, y=150, z=None, u=None, s=6000, param=None, wait=False)
2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) move_to(x=None, y=None, z=220, u=None, s=6000, param=None, wait=False)
2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) move_to(x=None, y=None, z=220, u=None, s=6000, param=None, wait=False)
2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) move_to(x=384.38, y=48.285, z=None, u=None, s=6000, param=None, wait=False)
2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) move_to(x=None, y=None, z=110, u=None, s=6000, param=None, wait=False)
2026-04-08 10:37:22 - [JubileeController]  - INFO - (SIMULATED) dwell(t=3000, millis=True)


🚀 Début de la distribution depuis A1...


NameError: name 'well_dest_name' is not defined

In [15]:
#index inoculateur
dict_outil = jm.get_tool_by_name("Inoculator")
idx_ino= dict_outil['index']


#changement d'outil
#changement d'outil
#jm.change_tool(ino, idx_ino)
jm.controller.pickup_tool_sequence(idx_ino)
jm.set_active_tool(idx_ino)
jm.set_tool_offset(idx_ino, (0, -31.5, 68))

inoculator = jm.tools_list[idx_ino]
inoculator.__class__ = Loop
inoculator._machine = jm.controller  # On le lie au contrôleur pour les mouvements
inoculator.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    inoculator.safe_z_movement = lambda: jm.controller.move_to(z=80)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{inoculator.name}' prêt (Classe: {type(inoculator).__name__})")

AttributeError: type object 'Loop' has no attribute 'name'

In [48]:
well_source = duckweed.labware.wells['A1'] ;
well_source.y = well_source.y - 31 + duckweed.coordinates[1] 
well_source.x = well_source.x + duckweed.coordinates[0] -7  #maj x
print(well_source)

Well A1 at coordinates (223.96, 205.535, 4.82)


In [ ]:
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# On définit la SOURCE fixe (ici slot A1 du slot 
slot_0 = jm.deck.get_slot('0')
jm.controller.move_to(z=200)



print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6

        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]

        well_dest.x = (well_dest.x + slot_0.coordinates[0] -7 )
        well_dest.y = (well_dest.y- 41 + slot_0.coordinates[1])#-41 ici car transferyt utilise move_to et pas effecteur to
        print(well_dest)

        print(f"Transfert de {well_source.name} vers {well_dest_name}...")

        try:
                # Appel du transfert (la sécurité safe_z est déjà injectée)
                jm.controller.move_to(z=200)        
                tool0.transfer(
                    source=well_source,      # FIXE
                    destination=well_dest,   # CHANGE à chaque itération
                    s=2000,
                    sweep_x=3,
                    sweep_y=3,
                    sweep_z=10,
                    sweep_speed=150,
                    up_speed=800,
                    randomize_pickup=True
                )
        except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break
                
        jm.controller.move_to(z=90)
        jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
        jm.controller.dwell(1000)
        detect.capture_octopi_image("apres")
        jm.controller.dwell(1000)

        compteur = 0      
        while(detect.detect_lens() == False and compteur < 3 ):
            compteur= compteur + 1
            jm.controller.move_to(z=200)
            try:
                    # Appel du transfert (la sécurité safe_z est déjà injectée)
                tool0.transfer(
                source=well_source,      # FIXE
                destination=well_dest,   # CHANGE à chaque itération
                s=2000,
                sweep_x=3,
                sweep_y=3,
                sweep_z=10,
                sweep_speed=150,
                up_speed=800,
                randomize_pickup=True
                    )
            except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break 
            jm.controller.move_to(z=90)
            jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
            jm.controller.dwell(5000)
            detect.capture_octopi_image("apres")
            jm.controller.dwell(1000)

  


print("✅ Distribution terminée sur toute la plaque !")

In [ ]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=200)
jm.controller.move_to(x=150, y=150)